# 01 - Image Classifier Training (MobileNetV2 Transfer Learning)

Trains the 5-class product classifier (Clothing, Shoes, Bags, Electronics, Groceries) using MobileNetV2 transfer learning, then exports `product_classifier.h5` into `app/models/`.

Dataset: a 5-class product image folder (e.g. Kaggle 'Retail Product Checkout Dataset' or a curated Fashion-MNIST-style set), organized as `data/products/<class_name>/*.jpg`.

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 8
DATA_DIR = '../data/products'  # class-per-subfolder image dataset

In [2]:
train_datagen = ImageDataGenerator(
    rescale=1./255, validation_split=0.2,
    rotation_range=15, zoom_range=0.15, horizontal_flip=True,
)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training'
)
val_gen = train_datagen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation'
)

Found 788 images belonging to 5 classes.
Found 197 images belonging to 5 classes.


In [3]:
base = tf.keras.applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet'
)
base.trainable = False

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(train_gen.num_classes, activation='softmax'),
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │             645 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,422,597 (9.24 MB)

 Trainable params: 164,613 (643.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [4]:
history = model.fit(train_gen, validation_data=val_gen, epochs=8)

Epoch 1/8
99/99 ━━━━━━━━━━━━━━━━━━━━ 35s 292ms/step - accuracy: 0.9061 - loss: 0.3135 - val_accuracy: 0.9543 - val_loss: 0.1463
Epoch 2/8
99/99 ━━━━━━━━━━━━━━━━━━━━ 25s 251ms/step - accuracy: 0.9822 - loss: 0.0553 - val_accuracy: 0.9746 - val_loss: 0.0930
Epoch 3/8
99/99 ━━━━━━━━━━━━━━━━━━━━ 25s 250ms/step - accuracy: 0.9835 - loss: 0.0452 - val_accuracy: 0.9848 - val_loss: 0.0799
Epoch 4/8
99/99 ━━━━━━━━━━━━━━━━━━━━ 25s 251ms/step - accuracy: 0.9797 - loss: 0.0626 - val_accuracy: 0.9797 - val_loss: 0.0826
Epoch 5/8
99/99 ━━━━━━━━━━━━━━━━━━━━ 25s 256ms/step - accuracy: 0.9962 - loss: 0.0121 - val_accuracy: 0.9645 - val_loss: 0.0820
Epoch 6/8
99/99 ━━━━━━━━━━━━━━━━━━━━ 27s 270ms/step - accuracy: 0.9975 - loss: 0.0091 - val_accuracy: 0.9746 - val_loss: 0.0964
Epoch 7/8
99/99 ━━━━━━━━━━━━━━━━━━━━ 25s 253ms/step - accuracy: 0.9975 - loss: 0.0102 - val_accuracy: 0.9695 - val_loss: 0.1047
Epoch 8/8
99/99 ━━━━━━━━━━━━━━━━━━━━ 26s 257ms/step - accuracy: 0.9822 - loss: 0.0372 - val_accuracy: 0.

In [5]:
loss, acc = model.evaluate(val_gen)
print(f'Validation accuracy: {acc*100:.2f}%')

model.save('../app/models/product_classifier.h5')
print('Saved to app/models/product_classifier.h5')

25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 219ms/step - accuracy: 0.9594 - loss: 0.1137


Validation accuracy: 95.94%
Saved to app/models/product_classifier.h5
